In [1]:
# =========================================================
# user settings
# =========================================================
EPOCHS = 300
EARLY_STOPPING_PATIENCE = 30
MIN_DELTA = 0.0
RANDOM_SEED = 42

BATCH_SIZE = 32
LEARNING_RATE = 1e-4
MODEL_NAME = "EDSR_HR_Aux"
NUM_RESBLK = 8
NUM_FEATURES = 128
HR_AUX_CHANNELS = 2
EDSR_RES_SCALE = 0.1

rcm_var = "tas"
gcm_name = "CanESM2"
rcm_name = "RCA4"
grid = "NAM-44i"
rcm_product = "raw"
factor = 4

train_start_year = 1951
train_end_year = 2005
val_start_year = 2006
val_end_year = 2099

DATA_ROOT = "/projects/sds-lab/Shuochen/downscaling/preprocessed"


# =========================================================
# imports
# =========================================================
import os
import json
import random

import matplotlib
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

from models import EDSR_HR_Aux

matplotlib.use("Agg")
import matplotlib.pyplot as plt


# =========================================================
# paths
# =========================================================
exp_folder_name = os.path.join(
    DATA_ROOT,
    f"{rcm_var}.{gcm_name}.{rcm_name}.day.{grid}.{rcm_product}.GCM_RCM",
)

input_file = "low_res.pth"       # LR GCM input
target_file = "high_res.pth"     # HR RCM target
hr_mask_file = "high_res_mask.pth"
hr_elev_file = "high_res_elevation.pth"

save_root = os.path.join(
    exp_folder_name,
    "trained_models",
    MODEL_NAME,
    "lr_gcm_with_hr_mask_elevation_to_hr_rcm_mse",
)

os.makedirs(save_root, exist_ok=True)

best_ckpt_path = os.path.join(save_root, "best_model.pth")
summary_path = os.path.join(save_root, "training_summary.json")
validation_plot_path = os.path.join(save_root, "validation_sample_prediction.png")
validation_summary_plot_path = os.path.join(save_root, "validation_set_summary.png")
validation_rmse_plot_path = os.path.join(save_root, "validation_set_rmse.png")


# =========================================================
# setup
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


# =========================================================
# helpers
# =========================================================
def assert_finite(tensor, name):
    if not torch.isfinite(tensor).all():
        raise ValueError(f"{name} contains NaN or inf values.")


def safe_std(tensor):
    std = tensor.std()
    if std.item() == 0:
        return torch.tensor(1.0, dtype=tensor.dtype)
    return std


def match_sample_dim(tensor, n_samples, name):
    if tensor.shape[0] == n_samples:
        return tensor
    if tensor.shape[0] == 1:
        print(f"{name} has one sample; expanding to {n_samples} samples.")
        return tensor.expand(n_samples, -1, -1, -1).contiguous()
    raise ValueError(
        f"{name} has incompatible sample dimension: "
        f"{tensor.shape[0]} vs expected {n_samples}"
    )


# =========================================================
# load data
# =========================================================
X = torch.load(os.path.join(exp_folder_name, input_file)).float()
y = torch.load(os.path.join(exp_folder_name, target_file)).float()
mask_hr = torch.load(os.path.join(exp_folder_name, hr_mask_file)).float()
elev_hr = torch.load(os.path.join(exp_folder_name, hr_elev_file)).float()

if X.ndim != 4:
    raise ValueError(f"Expected X to have shape [N, C, H, W], got {X.shape}")

if y.ndim != 4:
    raise ValueError(f"Expected y to have shape [N, C, H, W], got {y.shape}")

if mask_hr.ndim != 4:
    raise ValueError(f"Expected mask_hr to have shape [N, C, H, W], got {mask_hr.shape}")

if elev_hr.ndim != 4:
    raise ValueError(f"Expected elev_hr to have shape [N, C, H, W], got {elev_hr.shape}")

if X.shape[1] != 1:
    raise ValueError(f"Expected one LR GCM input channel, got {X.shape[1]}")

if y.shape[1] != 1:
    raise ValueError(f"Expected one HR RCM target channel, got {y.shape[1]}")

if mask_hr.shape[1] != 1:
    raise ValueError(f"Expected one HR mask channel, got {mask_hr.shape[1]}")

if elev_hr.shape[1] != 1:
    raise ValueError(f"Expected one HR elevation channel, got {elev_hr.shape[1]}")

if X.shape[0] != y.shape[0]:
    raise ValueError(f"Sample mismatch: X has {X.shape[0]}, y has {y.shape[0]}")

mask_hr = match_sample_dim(mask_hr, y.shape[0], "mask_hr")
elev_hr = match_sample_dim(elev_hr, y.shape[0], "elev_hr")

if mask_hr.shape != y.shape:
    raise ValueError(f"Mask shape mismatch: mask_hr has {mask_hr.shape}, y has {y.shape}")

if elev_hr.shape != y.shape:
    raise ValueError(f"Elevation shape mismatch: elev_hr has {elev_hr.shape}, y has {y.shape}")

mask_hr = (mask_hr > 0.5).float().contiguous()
elev_hr = torch.nan_to_num(elev_hr, nan=0.0, posinf=0.0, neginf=0.0).contiguous()
hr_aux = torch.cat([mask_hr, elev_hr], dim=1).contiguous()

if hr_aux.shape[1] != HR_AUX_CHANNELS:
    raise ValueError(f"Expected {HR_AUX_CHANNELS} HR auxiliary channels, got {hr_aux.shape[1]}")

if mask_hr.sum().item() == 0:
    raise ValueError("HR mask contains no land pixels.")

if y.shape[-2] != factor * X.shape[-2]:
    raise ValueError(
        f"Height mismatch: HR height {y.shape[-2]} != "
        f"{factor} * LR height {X.shape[-2]}"
    )

if y.shape[-1] != factor * X.shape[-1]:
    raise ValueError(
        f"Width mismatch: HR width {y.shape[-1]} != "
        f"{factor} * LR width {X.shape[-1]}"
    )

assert_finite(X, "X")
assert_finite(y, "y")
assert_finite(mask_hr, "mask_hr")
assert_finite(elev_hr, "elev_hr")
assert_finite(hr_aux, "hr_aux")

print("Loaded shapes:")
print("LR GCM X :", X.shape)
print("HR RCM y :", y.shape)
print("HR mask  :", mask_hr.shape)
print("HR elev  :", elev_hr.shape)
print("HR aux   :", hr_aux.shape)


# =========================================================
# train/validation split
# =========================================================
if train_end_year < train_start_year:
    raise ValueError("train_end_year must be >= train_start_year.")

if val_end_year < val_start_year:
    raise ValueError("val_end_year must be >= val_start_year.")

if gcm_name == "CanESM2":
    train_start_idx = 0
    train_end_idx = (train_end_year - train_start_year + 1) * 365
    val_start_idx = (val_start_year - train_start_year) * 365
    val_end_idx = (val_end_year - train_start_year + 1) * 365

elif gcm_name == "EC-EARTH":
    def is_leap(year):
        return (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)

    def days_between_years(start_year, end_year_inclusive):
        if end_year_inclusive < start_year:
            return 0
        total = 0
        for year in range(start_year, end_year_inclusive + 1):
            total += 366 if is_leap(year) else 365
        return total

    train_start_idx = 0
    train_end_idx = days_between_years(train_start_year, train_end_year)
    val_start_idx = days_between_years(train_start_year, val_start_year - 1)
    val_end_idx = days_between_years(train_start_year, val_end_year)

else:
    raise ValueError(f"Year split is not defined for gcm_name={gcm_name}")

if train_end_idx > X.shape[0]:
    raise ValueError(
        f"Training end index {train_end_idx} exceeds total samples {X.shape[0]}. "
        "Check train_start_year/train_end_year and the dataset calendar."
    )

if val_end_idx > X.shape[0]:
    raise ValueError(
        f"Validation end index {val_end_idx} exceeds total samples {X.shape[0]}. "
        "Check train_start_year/val_end_year and the dataset calendar."
    )

X_train = X[train_start_idx:train_end_idx]
y_train = y[train_start_idx:train_end_idx]
aux_train = hr_aux[train_start_idx:train_end_idx]

X_val = X[val_start_idx:val_end_idx]
y_val = y[val_start_idx:val_end_idx]
aux_val = hr_aux[val_start_idx:val_end_idx]
mask_val = mask_hr[val_start_idx:val_end_idx]

print("Year split:")
print(f"Train years: {train_start_year}-{train_end_year}")
print(f"Val years  : {val_start_year}-{val_end_year}")
print("train_start_idx:", train_start_idx)
print("train_end_idx  :", train_end_idx)
print("val_start_idx  :", val_start_idx)
print("val_end_idx    :", val_end_idx)

print("Split shapes:")
print("X_train  :", X_train.shape)
print("y_train  :", y_train.shape)
print("aux_train:", aux_train.shape)
print("X_val    :", X_val.shape)
print("y_val    :", y_val.shape)
print("aux_val  :", aux_val.shape)
print("mask_val :", mask_val.shape)

if X_train.shape[0] == 0 or X_val.shape[0] == 0:
    raise ValueError("Train or validation split is empty.")


# =========================================================
# normalization
# =========================================================
X_mean = X_train.mean()
X_std = safe_std(X_train)
y_mean = y_train.mean()
y_std = safe_std(y_train)

X_train_n = ((X_train - X_mean) / X_std).contiguous()
X_val_n = ((X_val - X_mean) / X_std).contiguous()
y_train_n = ((y_train - y_mean) / y_std).contiguous()
y_val_n = ((y_val - y_mean) / y_std).contiguous()

model_input_channels = X_train_n.shape[1]

print("Normalization:")
print("X_mean:", X_mean.item())
print("X_std :", X_std.item())
print("y_mean:", y_mean.item())
print("y_std :", y_std.item())
print("model_input_channels:", model_input_channels)
print("hr_aux_channels:", HR_AUX_CHANNELS)


# =========================================================
# datasets
# =========================================================
training_set = TensorDataset(X_train_n, aux_train, y_train_n)
validation_set = TensorDataset(X_val_n, aux_val, y_val_n)

train_dataloader = DataLoader(
    training_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_dataloader = DataLoader(
    validation_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
)


# =========================================================
# model
# =========================================================
model = EDSR_HR_Aux(
    num_resblk=NUM_RESBLK,
    num_features=NUM_FEATURES,
    input_channels=model_input_channels,
    output_channels=1,
    hr_aux_channels=HR_AUX_CHANNELS,
    scale=factor,
    res_scale=EDSR_RES_SCALE,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

scheduler = ReduceLROnPlateau(
    optimizer,
    patience=15,
    factor=0.5,
)

loss_fn = nn.MSELoss()


# =========================================================
# train
# =========================================================
train_loss_list = []
val_loss_list = []

best_val_loss = float("inf")
best_epoch = -1
epochs_no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss_sum = 0.0
    train_samples = 0

    for Xn, aux, yn in train_dataloader:
        Xn = Xn.to(device)
        aux = aux.to(device)
        yn = yn.to(device)

        y_pred_n = model(Xn, aux)
        loss = loss_fn(y_pred_n, yn)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = Xn.shape[0]
        train_loss_sum += loss.item() * batch_size
        train_samples += batch_size

    train_loss = train_loss_sum / train_samples
    train_loss_list.append(train_loss)

    model.eval()
    val_loss_sum = 0.0
    val_samples = 0

    with torch.no_grad():
        for Xn, aux, yn in val_dataloader:
            Xn = Xn.to(device)
            aux = aux.to(device)
            yn = yn.to(device)

            y_pred_n = model(Xn, aux)
            loss = loss_fn(y_pred_n, yn)

            batch_size = Xn.shape[0]
            val_loss_sum += loss.item() * batch_size
            val_samples += batch_size

    val_loss = val_loss_sum / val_samples
    val_loss_list.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_no_improve = 0

        torch.save(
            {
                "epoch": best_epoch,
                "model_name": MODEL_NAME,
                "input_file": input_file,
                "target_file": target_file,
                "hr_mask_file": hr_mask_file,
                "hr_elevation_file": hr_elev_file,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_loss": best_val_loss,
                "train_loss_list": train_loss_list,
                "val_loss_list": val_loss_list,
                "X_mean": X_mean,
                "X_std": X_std,
                "y_mean": y_mean,
                "y_std": y_std,
                "config": {
                    "rcm_var": rcm_var,
                    "gcm_name": gcm_name,
                    "rcm_name": rcm_name,
                    "grid": grid,
                    "rcm_product": rcm_product,
                    "factor": factor,
                    "train_start_year": train_start_year,
                    "train_end_year": train_end_year,
                    "val_start_year": val_start_year,
                    "val_end_year": val_end_year,
                    "model_name": MODEL_NAME,
                    "input_channels": model_input_channels,
                    "hr_aux_channels": HR_AUX_CHANNELS,
                    "num_resblk": NUM_RESBLK,
                    "num_features": NUM_FEATURES,
                    "res_scale": EDSR_RES_SCALE,
                    "batch_norm": False,
                    "loss": "mse",
                    "input": "LR GCM low_res.pth plus HR mask/elevation fused after upsampled prediction",
                    "target": "HR RCM high_res.pth only",
                    "hr_aux_description": "channel0=HR land-sea mask; channel1=HR elevation",
                    "regularization": None,
                },
            },
            best_ckpt_path,
        )
    else:
        epochs_no_improve += 1

    print(
        f"Epoch {epoch:03d} | "
        f"Train MSE: {train_loss:.6f} | "
        f"Val MSE: {val_loss:.6f} | "
        f"Best Val: {best_val_loss:.6f} | "
        f"Best Epoch: {best_epoch:03d} | "
        f"No improve: {epochs_no_improve}/{EARLY_STOPPING_PATIENCE} | "
        f"lr: {optimizer.param_groups[0]['lr']:.6e}"
    )

    if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break


# =========================================================
# summary
# =========================================================
summary = {
    "model_name": MODEL_NAME,
    "exp_folder_name": exp_folder_name,
    "save_root": save_root,
    "best_checkpoint": best_ckpt_path,
    "best_epoch": best_epoch,
    "best_val_loss": best_val_loss,
    "validation_sample_plot_path": validation_plot_path,
    "validation_summary_plot_path": validation_summary_plot_path,
    "validation_rmse_plot_path": validation_rmse_plot_path,
    "input_file": input_file,
    "target_file": target_file,
    "hr_mask_file": hr_mask_file,
    "hr_elevation_file": hr_elev_file,
    "input_channels": model_input_channels,
    "hr_aux_channels": HR_AUX_CHANNELS,
    "hr_aux_description": "channel0=HR land-sea mask; channel1=HR elevation",
    "res_scale": EDSR_RES_SCALE,
    "batch_norm": False,
    "train_start_year": train_start_year,
    "train_end_year": train_end_year,
    "val_start_year": val_start_year,
    "val_end_year": val_end_year,
    "loss": "mse",
    "regularization": None,
    "train_loss_list": train_loss_list,
    "val_loss_list": val_loss_list,
    "normalization": {
        "X_mean": X_mean.item(),
        "X_std": X_std.item(),
        "y_mean": y_mean.item(),
        "y_std": y_std.item(),
    },
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Training summary saved to:", summary_path)


# =========================================================
# plot one validation sample
# =========================================================
if len(validation_set) == 0:
    raise ValueError("Validation set is empty; cannot create validation sample plot.")

checkpoint = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

with torch.no_grad():
    X_sample = X_val_n[0:1].to(device)
    aux_sample = aux_val[0:1].to(device)
    y_true = y_val[0, 0].cpu().numpy()
    y_pred_n = model(X_sample, aux_sample)
    y_pred = (y_pred_n * y_std.to(device) + y_mean.to(device))[0, 0].cpu().numpy()

error = y_pred - y_true

value_pixels = np.concatenate([
    y_true[np.isfinite(y_true)].ravel(),
    y_pred[np.isfinite(y_pred)].ravel(),
])

if value_pixels.size == 0:
    raise ValueError("No finite pixels found for validation plot.")

vmin = float(value_pixels.min())
vmax = float(value_pixels.max())
err_abs = float(np.nanmax(np.abs(error)))
if err_abs == 0:
    err_abs = 1.0

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

im0 = axes[0].imshow(y_true, origin="lower", cmap="viridis", vmin=vmin, vmax=vmax)
axes[0].set_title("HR RCM target")
axes[0].set_xticks([])
axes[0].set_yticks([])
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(y_pred, origin="lower", cmap="viridis", vmin=vmin, vmax=vmax)
axes[1].set_title("EDSR prediction")
axes[1].set_xticks([])
axes[1].set_yticks([])
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(error, origin="lower", cmap="coolwarm", vmin=-err_abs, vmax=err_abs)
axes[2].set_title("Prediction - target")
axes[2].set_xticks([])
axes[2].set_yticks([])
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

fig.suptitle("LR GCM to HR RCM, EDSR first validation sample")
fig.savefig(validation_plot_path, dpi=200, bbox_inches="tight")
plt.close(fig)

print("Validation sample plot saved to:", validation_plot_path)


# =========================================================
# plot entire validation set summary
# =========================================================
sum_true = None
sum_pred = None
sum_error = None
sum_sq_error = None
total_samples = 0
full_se_sum = 0.0
full_pixel_count = 0
land_se_sum = 0.0
land_pixel_count = 0.0
val_offset = 0

with torch.no_grad():
    for Xn, aux, yn in val_dataloader:
        Xn = Xn.to(device)
        aux = aux.to(device)
        yn = yn.to(device)

        y_pred_n = model(Xn, aux)
        y_pred_batch = (y_pred_n * y_std.to(device) + y_mean.to(device)).cpu()
        y_true_batch = (yn * y_std.to(device) + y_mean.to(device)).cpu()
        error_batch = y_pred_batch - y_true_batch
        squared_error_batch = error_batch ** 2

        batch_size = Xn.shape[0]
        mask_batch = mask_val[val_offset: val_offset + batch_size].cpu()
        val_offset += batch_size

        full_se_sum += squared_error_batch.sum().item()
        full_pixel_count += squared_error_batch.numel()
        land_se_sum += (squared_error_batch * mask_batch).sum().item()
        land_pixel_count += mask_batch.sum().item()

        batch_sum_true = y_true_batch.sum(dim=0)
        batch_sum_pred = y_pred_batch.sum(dim=0)
        batch_sum_error = error_batch.sum(dim=0)
        batch_sum_sq_error = squared_error_batch.sum(dim=0)

        if sum_true is None:
            sum_true = batch_sum_true
            sum_pred = batch_sum_pred
            sum_error = batch_sum_error
            sum_sq_error = batch_sum_sq_error
        else:
            sum_true += batch_sum_true
            sum_pred += batch_sum_pred
            sum_error += batch_sum_error
            sum_sq_error += batch_sum_sq_error

        total_samples += Xn.shape[0]

if total_samples <= 0:
    raise ValueError("Validation set is empty; cannot create validation summary plot.")

if full_pixel_count <= 0:
    raise ValueError("Validation set has no pixels for full-image MSE.")

if land_pixel_count <= 0:
    raise ValueError("Validation HR mask contains no land pixels.")

val_physical_mse_full_image = full_se_sum / full_pixel_count
val_physical_mse_land_only = land_se_sum / land_pixel_count

summary["validation_physical_mse_full_image"] = val_physical_mse_full_image
summary["validation_physical_mse_land_only"] = val_physical_mse_land_only

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

mean_true = (sum_true / total_samples)[0].numpy()
mean_pred = (sum_pred / total_samples)[0].numpy()
bias = mean_true - mean_pred
rmse = torch.sqrt(sum_sq_error / total_samples)[0].numpy()

summary_values = np.concatenate([
    mean_true[np.isfinite(mean_true)].ravel(),
    mean_pred[np.isfinite(mean_pred)].ravel(),
])

if summary_values.size == 0:
    raise ValueError("No finite pixels found for validation summary plot.")

summary_vmin = float(summary_values.min())
summary_vmax = float(summary_values.max())
bias_abs = float(np.nanmax(np.abs(bias)))
if bias_abs == 0:
    bias_abs = 1.0

rmse_vmax = float(np.nanmax(rmse))
if rmse_vmax == 0:
    rmse_vmax = 1.0

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

im0 = axes[0].imshow(mean_true, origin="lower", cmap="viridis", vmin=summary_vmin, vmax=summary_vmax)
axes[0].set_title("Mean HR RCM target")
axes[0].set_xticks([])
axes[0].set_yticks([])
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(mean_pred, origin="lower", cmap="viridis", vmin=summary_vmin, vmax=summary_vmax)
axes[1].set_title("Mean EDSR prediction")
axes[1].set_xticks([])
axes[1].set_yticks([])
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(bias, origin="lower", cmap="coolwarm", vmin=-bias_abs, vmax=bias_abs)
axes[2].set_title("Bias: target - prediction")
axes[2].set_xticks([])
axes[2].set_yticks([])
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

fig.suptitle("LR GCM to HR RCM, EDSR validation mean fields")
fig.savefig(validation_summary_plot_path, dpi=200, bbox_inches="tight")
plt.close(fig)

print("Validation summary plot saved to:", validation_summary_plot_path)

fig, ax = plt.subplots(1, 1, figsize=(6, 4), constrained_layout=True)
im = ax.imshow(rmse, origin="lower", cmap="magma", vmin=0.0, vmax=rmse_vmax)
ax.set_title("Validation set RMSE")
ax.set_xticks([])
ax.set_yticks([])
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("LR GCM to HR RCM, EDSR validation set")
fig.savefig(validation_rmse_plot_path, dpi=200, bbox_inches="tight")
plt.close(fig)

print("Validation RMSE plot saved to:", validation_rmse_plot_path)
print("Best checkpoint saved to:", best_ckpt_path)
print(f"Best validation MSE, normalized: {best_val_loss:.6f}")
print(f"Validation physical MSE, full image: {val_physical_mse_full_image:.6f}")
print(f"Validation physical MSE, land only: {val_physical_mse_land_only:.6f}")
print("Training summary updated with final validation physical MSEs:", summary_path)


Using device: cuda
Loaded shapes:
LR GCM X : torch.Size([54750, 1, 13, 29])
HR RCM y : torch.Size([54750, 1, 52, 116])
HR mask  : torch.Size([54750, 1, 52, 116])
HR elev  : torch.Size([54750, 1, 52, 116])
HR aux   : torch.Size([54750, 2, 52, 116])
Year split:
Train years: 1951-2005
Val years  : 2006-2099
train_start_idx: 0
train_end_idx  : 20075
val_start_idx  : 20075
val_end_idx    : 54385
Split shapes:
X_train  : torch.Size([20075, 1, 13, 29])
y_train  : torch.Size([20075, 1, 52, 116])
aux_train: torch.Size([20075, 2, 52, 116])
X_val    : torch.Size([34310, 1, 13, 29])
y_val    : torch.Size([34310, 1, 52, 116])
aux_val  : torch.Size([34310, 2, 52, 116])
mask_val : torch.Size([34310, 1, 52, 116])
Normalization:
X_mean: 7.1071882247924805
X_std : 11.00319766998291
y_mean: 5.909543037414551
y_std : 10.123307228088379
model_input_channels: 1
hr_aux_channels: 2
Epoch 000 | Train MSE: 0.106164 | Val MSE: 0.071272 | Best Val: 0.071272 | Best Epoch: 000 | No improve: 0/30 | lr: 1.000000e-04
